# Structuration automatique des `<history>` et `<handNote>` dans les `msDesc`

**Pipeline** : XML (teiCorpus) → sous-ensemble ciblé du msDesc → API Anthropic → injection de `<history>` et `<handNote>` → XML enrichi

**Fichiers attendus :**
- `OAK.txt` — clé API Anthropic
- `GLOBAL_PROMPT.txt` — system prompt
- `examples.xml` — exemples avant/après pour le few-shot *(optionnel)*
- `input.xml` — fichier TEI source

In [1]:
# Décommenter si nécessaire
# %pip install anthropic lxml tqdm
# %pip install ipywidgets

In [2]:
import copy
import datetime
import time
import concurrent.futures
from pathlib import Path
from threading import Lock

import anthropic
from lxml import etree
from tqdm.auto import tqdm

## Configuration

In [3]:
# ── Fichiers ──────────────────────────────────────────────────────────────────
KEY_FILE      = Path("OAK.txt")
PROMPT_FILE   = Path("GLOBAL_PROMPT.txt")
EXAMPLES_FILE = Path("examples.xml")           # Mettre None pour désactiver
XML_INPUT     = Path("../../catalogues/modified_CMDF_1enr.xml")
XML_OUTPUT    = Path("../../catalogues/modified_CMDF_1enr_history_structured.xml")

# ── Paramètres API ────────────────────────────────────────────────────────────
MODEL       = "claude-sonnet-4-6"   # Sonnet : 5x moins cher qu'Opus, suffisant pour structuration
MAX_TOKENS  = 2048
MAX_WORKERS = 3                     # Requêtes parallèles
RETRY_MAX   = 3                     # Tentatives sur erreur transitoire
RETRY_DELAY = 5                     # Secondes entre tentatives

# ── Éléments du msDesc envoyés à l'API (les autres sont ignorés) ──────────────
# msIdentifier : contexte d'identification (dépôt, cote)
# history      : contenu existant à compléter/restructurer
# physDesc     : pour lire les <handNote/> vides et le contexte matériel
INPUT_ELEMENTS = ["msIdentifier", "history", "physDesc"]

# ── Lecture des fichiers externes ─────────────────────────────────────────────
API_KEY       = KEY_FILE.read_text(encoding="utf-8").strip()
GLOBAL_PROMPT = PROMPT_FILE.read_text(encoding="utf-8").strip()

print(f"Clé API    : {API_KEY[:8]}{'*' * (len(API_KEY) - 8)}")
print(f"Prompt     : {len(GLOBAL_PROMPT)} caractères")
print(f"Exemples   : {EXAMPLES_FILE if EXAMPLES_FILE and EXAMPLES_FILE.exists() else 'non chargés'}")
print(f"Modèle     : {MODEL}")

Clé API    : sk-ant-a****************************************************************************************************
Prompt     : 8352 caractères
Exemples   : examples.xml
Modèle     : claude-sonnet-4-6


## Chargement des exemples few-shot (optionnel)

Le fichier `examples.xml` contient des paires avant/après.
Chaque exemple est injecté comme paire `user` / `assistant` avant le message réel,
ce qui est plus efficace que de les mettre dans le system prompt.

Structure attendue :
```xml
<examples>
  <example>
    <input><!-- msDesc allégé envoyé à l'API --></input>
    <output><!-- <result> attendu en réponse --></output>
  </example>
</examples>
```

In [4]:
def load_few_shot_messages(examples_path: Path | None) -> list[dict]:
    """
    Charge les exemples depuis examples.xml et les convertit en paires
    de messages user/assistant pour l'API Anthropic.
    Ces messages sont prépendés à chaque appel, avant le msDesc réel.
    """
    if not examples_path or not examples_path.exists():
        return []
    ex_tree = etree.parse(str(examples_path), etree.XMLParser(recover=True))
    
    messages = []
    for ex in ex_tree.findall("example"):
        input_el  = ex.find("input")
        output_el = ex.find("output")
        if input_el is None or output_el is None:
            continue
        # Sérialiser le contenu (enfants directs, pas la balise <input> elle-même)
        input_xml  = "".join(etree.tostring(c, encoding="unicode") for c in input_el)
        output_xml = "".join(etree.tostring(c, encoding="unicode") for c in output_el)
        messages.append({"role": "user",      "content": input_xml.strip()})
        messages.append({"role": "assistant", "content": output_xml.strip()})
    print(f"{len(messages) // 2} exemple(s) few-shot chargé(s).")
    return messages

FEW_SHOT_MESSAGES = load_few_shot_messages(EXAMPLES_FILE)

12 exemple(s) few-shot chargé(s).


## Chargement du XML

In [5]:
TEI_NS = "http://www.tei-c.org/ns/1.0"

parser = etree.XMLParser(remove_blank_text=False, ns_clean=False)
tree   = etree.parse(str(XML_INPUT), parser)
root   = tree.getroot()

doc_ns = root.nsmap.get(None) or root.nsmap.get("tei") or ""
print(f"Namespace détecté : {doc_ns or '(aucun)'}")

def tag(local: str) -> str:
    """Retourne le nom qualifié Clark pour un tag local."""
    return f"{{{doc_ns}}}{local}" if doc_ns else local

MSDES_TAG   = tag("msDesc")
HISTORY_TAG = tag("history")
PHYSDESC_TAG = tag("physDesc")
HANDDESC_TAG = tag("handDesc")
HANDNOTE_TAG = tag("handNote")

ms_nodes = root.findall(f".//{MSDES_TAG}")
print(f"{len(ms_nodes)} éléments <msDesc> trouvés.")

Namespace détecté : http://www.tei-c.org/ns/1.0
1124 éléments <msDesc> trouvés.


In [6]:
# Aperçu de la première notice et de son sous-ensemble envoyé à l'API
if ms_nodes:
    print("=== msDesc complet (début) ===")
    print(etree.tostring(ms_nodes[0], pretty_print=True, encoding="unicode")[:600])

=== msDesc complet (début) ===
<msDesc xmlns="http://www.tei-c.org/ns/1.0" xmlns:functx="http://www.functx.com">
                        <msIdentifier>
                           <settlement xml:id="d1e19460">Chantilly</settlement>
                           <repository>Mus. Condé</repository>
                           <idno>9</idno>
                           <altIdentifier>
                              <idno>1695</idno>
                           </altIdentifier>
                        </msIdentifier>
                        <msContents>
                           <summary>Psalterium</summary>
                        <


## Fonctions XML

In [7]:
def build_api_input(ms_node: etree._Element) -> str:
    """
    Construit un <msDesc> allégé :
      - msIdentifier, history, physDesc  (définis dans INPUT_ELEMENTS)
      - additional/adminInfo/note        (extraction ciblée pour éviter
                                          de transmettre listBibl, etc.)
    """
    slim = etree.Element(ms_node.tag, attrib=ms_node.attrib, nsmap=ms_node.nsmap)

    # Éléments standard
    for local_name in INPUT_ELEMENTS:
        for child in ms_node.findall(tag(local_name)):
            slim.append(copy.deepcopy(child))

    # Notes de additional/adminInfo uniquement
    additional = ms_node.find(tag("additional"))
    if additional is not None:
        admin_info = additional.find(tag("adminInfo"))
        if admin_info is not None:
            notes = admin_info.findall(tag("note"))
            if notes:
                slim_add   = etree.SubElement(slim, tag("additional"))
                slim_admin = etree.SubElement(slim_add, tag("adminInfo"))
                for note in notes:
                    slim_admin.append(copy.deepcopy(note))

    return etree.tostring(slim, pretty_print=True, encoding="unicode")

# Vérification : aperçu de l'entrée allégée pour la première notice
if ms_nodes:
    print("=== Entrée allégée envoyée à l'API ===")
    print(build_api_input(ms_nodes[0])[:600])

=== Entrée allégée envoyée à l'API ===
<msDesc xmlns="http://www.tei-c.org/ns/1.0" xmlns:functx="http://www.functx.com"><msIdentifier>
                           <settlement xml:id="d1e19460">Chantilly</settlement>
                           <repository>Mus. Condé</repository>
                           <idno>9</idno>
                           <altIdentifier>
                              <idno>1695</idno>
                           </altIdentifier>
                        </msIdentifier>
                        <history>
                           <origin>
                              <origPlace xml:id="d1e19511">Paris</origPlac


In [8]:
def normalise_namespace(element: etree._Element) -> etree._Element:
    """
    Réécrit récursivement les tags sans namespace pour qu'ils utilisent
    le namespace du document source (doc_ns).
    """
    def _fix(el):
        if doc_ns and not etree.QName(el).namespace:
            el.tag = f"{{{doc_ns}}}{etree.QName(el).localname}"
        for child in el:
            _fix(child)
    el = copy.deepcopy(element)
    _fix(el)
    return el


def extract_result(response_text: str) -> dict:
    """
    Parse la réponse de l'API et retourne :
      {
        'history'  : etree._Element | None,
        'handnotes': list[etree._Element]     # peut être vide
      }
    La réponse attendue est un <result> contenant un <history>
    et, optionnellement, un ou plusieurs <handNote> frères.
    Les <handNote> situés à l'intérieur de <history> sont ignorés
    (ils n'y ont pas leur place en TEI).
    """
    text = response_text.strip()
    # Supprimer les blocs Markdown ```xml … ```
    if text.startswith("```"):
        lines = text.splitlines()
        text = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:]).strip()

    def _local(el):   return etree.QName(el).localname
    def _is(el, name): return _local(el) == name

    def _ancestors_include_history(el):
        p = el.getparent()
        while p is not None:
            if _is(p, "history"):
                return True
            p = p.getparent()
        return False

    def _parse_and_route(root_el) -> dict:
        history  = None
        handnotes = []
        # Si la racine est elle-même <history>
        if _is(root_el, "history"):
            return {"history": root_el, "handnotes": []}
        # Chercher <history> et <handNote> dans l'arbre
        for el in root_el.iter():
            if _is(el, "history") and history is None:
                history = el
            elif _is(el, "handNote") and not _ancestors_include_history(el):
                handnotes.append(el)
        return {"history": history, "handnotes": handnotes}

    ns_decl = f' xmlns="{doc_ns}"' if doc_ns else ""
    for candidate in [text, f"<_root{ns_decl}>{text}</_root>"]:
        try:
            el = etree.fromstring(candidate.encode("utf-8"))
            res = _parse_and_route(el)
            if res["history"] is not None or res["handnotes"]:
                return res
        except etree.XMLSyntaxError:
            continue
    return {"history": None, "handnotes": []}


def inject_history(ms_node: etree._Element, history_el: etree._Element) -> None:
    """
    Remplace le <history> existant dans ms_node (enfant direct uniquement)
    ou l'ajoute s'il est absent.
    En TEI, <history> est enfant direct de <msDesc> ; on évite la recherche
    profonde qui risquerait de toucher des <history> dans d'autres contextes.
    """
    existing = ms_node.find(HISTORY_TAG)
    new_history = normalise_namespace(history_el)
    if existing is not None:
        existing.getparent().replace(existing, new_history)
    else:
        ms_node.append(new_history)


def inject_handnotes(ms_node: etree._Element, handnote_els: list) -> int:
    """
    Injecte les <handNote> dans physDesc/handDesc.
    Stratégie de correspondance :
      1. Par @xml:id si les handNote reçus portent cet attribut
      2. Par ordre d'apparition sinon
    Un <handNote> vide = pas de texte significatif, pas d'enfants.
    Retourne le nombre de <handNote> injectés.
    """
    if not handnote_els:
        return 0

    phys_desc = ms_node.find(PHYSDESC_TAG)
    if phys_desc is None:
        return 0   # Pas de physDesc : on n'en crée pas sans contexte

    hand_desc = phys_desc.find(HANDDESC_TAG)
    if hand_desc is None:
        hand_desc = etree.SubElement(phys_desc, HANDDESC_TAG)

    XML_ID = "{http://www.w3.org/XML/1998/namespace}id"

    def _is_empty(el):
        return not (el.text or "").strip() and len(el) == 0

    existing_hn = hand_desc.findall(HANDNOTE_TAG)
    empty_by_id  = {hn.get(XML_ID): hn for hn in existing_hn if _is_empty(hn) and hn.get(XML_ID)}
    empty_slots  = [hn for hn in existing_hn if _is_empty(hn) and not hn.get(XML_ID)]

    injected = 0
    for new_hn in handnote_els:
        normalised = normalise_namespace(new_hn)
        hn_id = new_hn.get(XML_ID)
        if hn_id and hn_id in empty_by_id:
            # Correspondance par xml:id
            slot = empty_by_id.pop(hn_id)
            slot.getparent().replace(slot, normalised)
        elif empty_slots:
            # Correspondance par ordre
            slot = empty_slots.pop(0)
            slot.getparent().replace(slot, normalised)
        else:
            # Aucun slot vide : on appende
            hand_desc.append(normalised)
        injected += 1

    return injected

In [9]:
client = anthropic.Anthropic(api_key=API_KEY)


def call_api(user_content: str) -> str:
    """
    Appel API avec few-shot et relance sur erreur transitoire.
    Les exemples few-shot sont prépendés comme paires user/assistant.
    """
    messages = FEW_SHOT_MESSAGES + [{"role": "user", "content": user_content}]
    last_error = None
    for attempt in range(1, RETRY_MAX + 1):
        try:
            message = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=GLOBAL_PROMPT,
                messages=messages,
            )
            return message.content[0].text
        except anthropic.RateLimitError:
            last_error = f"Rate limit (tentative {attempt}/{RETRY_MAX})"
            time.sleep(RETRY_DELAY * attempt)
        except anthropic.APIStatusError as e:
            last_error = f"API error {e.status_code}: {e.message}"
            if e.status_code >= 500:
                time.sleep(RETRY_DELAY)
            else:
                break
        except Exception as e:
            last_error = str(e)
            break
    raise RuntimeError(last_error)


def process_one(ms_node: etree._Element) -> dict:
    """
    Pour une notice :
      1. Construit l'entrée allégée (msIdentifier + history + physDesc)
      2. Appelle l'API
      3. Extrait <history> et éventuels <handNote>
    """
    ms_id = (ms_node.get("{http://www.w3.org/XML/1998/namespace}id")
              or ms_node.get("n") or ms_node.get("id") or "(sans id)")

    ts = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] → {ms_id}", flush=True)

    ms_xml = build_api_input(ms_node)

    try:
        response_text = call_api(ms_xml)
    except RuntimeError as e:
        print(f"[{datetime.datetime.now():%H:%M:%S}] ✗ {ms_id} — {e}", flush=True)
        return {"id": ms_id, "ok": False, "error": str(e)}

    result = extract_result(response_text)

    if result["history"] is None:
        print(f"[{datetime.datetime.now():%H:%M:%S}] ✗ {ms_id} — pas de <history>", flush=True)
        return {"id": ms_id, "ok": False,
                "error": "Aucun <history> dans la réponse.",
                "raw": response_text[:300]}

    n_hn = len(result["handnotes"])
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    hn_info = f" +{n_hn} handNote" if n_hn else ""
    print(f"[{ts}] ✓ {ms_id}{hn_info}", flush=True)

    return {"id": ms_id, "ok": True,
            "history": result["history"],
            "handnotes": result["handnotes"]}

## Traitement en lot

In [ ]:
# ── Paramètre de sauvegarde intermédiaire ─────────────────────────────────────
BATCH_SIZE = 10   # Export XML toutes les N notices traitées

results   = []
tree_lock = Lock()



def process_and_inject(ms_node: etree._Element) -> dict:
    res = process_one(ms_node)
    if res["ok"]:
        with tree_lock:
            inject_history(ms_node, res.pop("history"))
            n_hn = inject_handnotes(ms_node, res.pop("handnotes"))
            res["handnotes_injected"] = n_hn
    return res



def save_checkpoint(n_done: int) -> None:
    """Écrit le XML courant sur disque. Appelé après chaque lot."""
    tree.write(str(XML_OUTPUT), pretty_print=True,
               xml_declaration=True, encoding="UTF-8")
    size_kb = XML_OUTPUT.stat().st_size / 1024
    print(f"  💾 checkpoint — {n_done}/{len(ms_nodes)} notices"
          f" ({size_kb:.0f} ko)", flush=True)


# Découper ms_nodes en lots de BATCH_SIZE
chunks = [ms_nodes[i:i + BATCH_SIZE]
          for i in range(0, len(ms_nodes), BATCH_SIZE)]

with tqdm(total=len(ms_nodes), desc="Notices") as pbar:
    for chunk_idx, chunk in enumerate(chunks):

        # Traitement parallèle du lot courant
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(process_and_inject, node): node
                       for node in chunk}
            for future in concurrent.futures.as_completed(futures):
                res = future.result()
                results.append(res)
                hn_info = f" +{res.get('handnotes_injected', 0)}hn" \
                          if res.get("ok") else ""
                pbar.set_postfix_str(
                    f"{'✓' if res['ok'] else '✗'} {res['id']}{hn_info}")
                pbar.update(1)

        # Sauvegarde après chaque lot
        save_checkpoint(len(results))

ok_count  = sum(1 for r in results if r["ok"])
hn_count  = sum(r.get("handnotes_injected", 0) for r in results)
err_count = len(results) - ok_count
print(f"\nTerminé : {ok_count} OK ({hn_count} handNote injectés),"
      f" {err_count} erreur(s)")




[14:59:56] → (sans id)
[14:59:56] → (sans id)
[14:59:56] → (sans id)


Notices:   0%|          | 0/1124 [00:00<?, ?it/s]

[15:00:02] ✓ (sans id)
[15:00:02] → (sans id)
[15:00:04] ✓ (sans id) +1 handNote
[15:00:09] ✓ (sans id) +1 handNote
[15:00:11] ✓ (sans id)

Terminé : 4 OK (2 handNote injectés), 0 erreur(s)
Fichier écrit : ..\..\catalogues\modified_CMDF_1enr_history_structured.xml  (7435.3 ko)
=== Premier <history> ===
<history xmlns="http://www.tei-c.org/ns/1.0" xmlns:functx="http://www.functx.com">
    <origin>
      <origPlace xml:id="d1e19511">Paris</origPlace>
      <origDate xml:id="d1e19513">[avant 1213 ?]</origDate>
      <p corresp="recipient">
        <desc>Ingeburge de Danemark, reine de France en 1193, morte en 1236 (plusieurs mentions
          contemporaines du texte ont été ajoutées au calendrier : obits du roi Waldemar et de la
          reine Sophie, parents d'Ingeburge, et de son amie Eléonore de Vermandois, morte en 1213;
          d'une autre main, à la date du 27 juillet :</desc>
        <quote>Sexto kalendas augusti anno Domini M°CC° quarto decimo, veinqui Phelippe, li rois de
   

In [11]:
# Cellule de monitoring : exécuter pendant le traitement pour voir l'avancement
# (fonctionne uniquement si le kernel autorise deux cellules simultanées,
#  sinon attendre la fin ou utiliser les prints horodatés)
done = len(results)
print(f"Traitées : {done} / {len(ms_nodes)}")
if done > 0:
    print(f"OK       : {sum(1 for r in results if r['ok'])}")
    print(f"Erreurs  : {sum(1 for r in results if not r['ok'])}")

Traitées : 4 / 1124
OK       : 4
Erreurs  : 0


In [12]:
# Rapport détaillé des erreurs
errors = [r for r in results if not r["ok"]]
if errors:
    print(f"{len(errors)} notice(s) en erreur :")
    for r in errors:
        print(f"  [{r['id']}] {r.get('error', '')}")
        if "raw" in r:
            print(f"    Début réponse : {r['raw']}")
else:
    print("Aucune erreur.")

Aucune erreur.


In [13]:
# Relancer uniquement les notices en erreur
# (décommenter et exécuter après avoir corrigé le problème)

# failed_ids = {r["id"] for r in results if not r["ok"]}
# XML_ID_ATTR = "{http://www.w3.org/XML/1998/namespace}id"
# to_retry = [
#     n for n in ms_nodes
#     if (n.get(XML_ID_ATTR) or n.get("n") or n.get("id") or "(sans id)") in failed_ids
# ]
# print(f"{len(to_retry)} notice(s) à relancer")
# with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
#     futures = {executor.submit(process_and_inject, node): node for node in to_retry}
#     for future in concurrent.futures.as_completed(futures):
#         res = future.result()
#         # Mettre à jour results : remplacer l'entrée en erreur
#         results = [r for r in results if r["id"] != res["id"]]
#         results.append(res)
#         print(f"{'✓' if res['ok'] else '✗'} {res['id']}")

## Export du XML enrichi